# EEG Aggregation — Step 1: Channel Averaging

**Purpose:** aggregate channel-level EEG power into participant-level global summaries by task, session, condition, and band.

**Unit of analysis:** participant. The 32 channels are averaged within each participant recording block; they are not treated as independent observations. Mathematics and Reading remain separate, task conditions and bands remain separate, and absolute/relative power are kept as distinct outcomes. No statistical hypothesis tests or ROI analysis are performed in this notebook. Missing PRE/POST pairs are not imputed.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

FILES = {
    "Math Control": "eeg_analysis_results_math_control.csv",
    "Math Experiment": "eeg_analysis_results_math_experiment.csv",
    "Reading Control": "eeg_analysis_results_read_control.csv",
    "Reading Experiment": "eeg_analysis_results_read_experiment.csv",
}
EXPECTED_CHANNELS, EXPECTED_BANDS = 32, 5
EXPECTED_ROWS = EXPECTED_CHANNELS * EXPECTED_BANDS
EXPECTED_BAND_SET = {"alpha", "beta", "delta", "gamma", "theta"}

# Optional: specify the source CSV directory manually if automatic discovery fails.
DATA_DIR_OVERRIDE = None

search_roots = [Path.cwd(), Path.cwd().parent, Path("/mnt/data")]
candidates = []
for root in search_roots:
    if not root.exists():
        continue
    candidates.extend([root, root / "Results", root / "analysis" / "results" / "eeg", root / "analysis" / "results"])
    # Find common result-folder names, but avoid scanning hidden folders and .git.
    candidates.extend(p for p in root.glob("**/Results") if p.is_dir() and ".git" not in p.parts)
    candidates.extend(p for p in root.glob("**/results") if p.is_dir() and ".git" not in p.parts)
# Deduplicate paths without changing priority.
seen = set()
candidates = [p for p in candidates if not (str(p.resolve()) in seen or seen.add(str(p.resolve())))]

if DATA_DIR_OVERRIDE is not None:
    DATA_DIR = Path(DATA_DIR_OVERRIDE).expanduser()
    if not all((DATA_DIR / fname).is_file() for fname in FILES.values()):
        raise FileNotFoundError(f"DATA_DIR_OVERRIDE does not contain all four source CSV files: {DATA_DIR}")
else:
    DATA_DIR = next((p for p in candidates if p.is_dir() and all((p / fname).is_file() for fname in FILES.values())), None)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate all four EEG source CSV files. Set DATA_DIR_OVERRIDE to their folder.\n"
        + "Required files: " + ", ".join(FILES.values())
    )

OUTPUT_DIR = Path.cwd() / "outputs" / "eeg_aggregation_step1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Inputs:", DATA_DIR.resolve())
print("Outputs:", OUTPUT_DIR.resolve())

Inputs: /mnt/data
Outputs: /mnt/data/outputs/eeg_aggregation_step1


## 1. Load source tables and validate schema

In [2]:
required = {"subject_id", "session", "run", "condition", "channel", "band", "absolute_power", "relative_power"}
parts = []
for task_group, filename in FILES.items():
    frame = pd.read_csv(DATA_DIR / filename)
    missing_cols = required - set(frame.columns)
    if missing_cols:
        raise ValueError(f"{filename}: missing columns {sorted(missing_cols)}")
    task, group = task_group.split(" ", 1)
    frame["task"], frame["group"] = task, group
    frame["timepoint"] = frame["session"].map({"ses-1": "PRE", "ses-2": "POST"})
    if frame["timepoint"].isna().any():
        bad_sessions = sorted(frame.loc[frame["timepoint"].isna(), "session"].astype(str).unique())
        raise ValueError(f"{filename}: unrecognized session labels: {bad_sessions}")
    frame["band"] = frame["band"].astype(str).str.strip().str.lower()
    frame["channel"] = frame["channel"].astype(str).str.strip()
    for col in ["absolute_power", "relative_power"]:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
    if not np.isfinite(frame[["absolute_power", "relative_power"]].to_numpy()).all():
        raise ValueError(f"{filename}: non-numeric, missing, or infinite power values found")
    if not frame["band"].isin(EXPECTED_BAND_SET).all():
        bad_bands = sorted(frame.loc[~frame["band"].isin(EXPECTED_BAND_SET), "band"].unique())
        raise ValueError(f"{filename}: unexpected band labels: {bad_bands}")
    parts.append(frame)
source = pd.concat(parts, ignore_index=True)
print(f"Combined source: {source.shape[0]:,} rows")
display(source.head())

Combined source: 51,200 rows


,subject_id,session,run,condition,sampling_frequency,duration,channel_count,sample_count,channel,band,frequency_min,frequency_max,absolute_power,relative_power,event_id,event_onset,event_duration,event_code,task,group,timepoint
0,Mc02,ses-1,1,Baseline,256.0,179.996094,32,46080,C3,alpha,8.0,13.0,3659.015418,3.616653,NaN,NaN,NaN,NaN,Math,Control,PRE
1,Mc02,ses-1,1,Baseline,256.0,179.996094,32,46080,C3,beta,13.0,30.0,2877.068650,2.843759,NaN,NaN,NaN,NaN,Math,Control,PRE
2,Mc02,ses-1,1,Baseline,256.0,179.996094,32,46080,C3,delta,0.5,4.0,86743.462250,85.739176,NaN,NaN,NaN,NaN,Math,Control,PRE
3,Mc02,ses-1,1,Baseline,256.0,179.996094,32,46080,C3,gamma,30.0,50.0,781.934466,0.772881,NaN,NaN,NaN,NaN,Math,Control,PRE
4,Mc02,ses-1,1,Baseline,256.0,179.996094,32,46080,C3,theta,4.0,8.0,7109.846515,7.027531,NaN,NaN,NaN,NaN,Math,Control,PRE


## 2. QC by participant/session/condition block
Each block is expected to contain exactly 160 unique channel-band combinations (32 channels × 5 bands), with no missing power, no duplicate keys, and relative band percentages summing to 100% for each channel. QC is explicit about unique combinations, not just row/channel/band counts.

In [3]:
qc_rows = []
block_keys = ["subject_id", "group", "task", "session", "timepoint", "run", "condition"]
for keys, block in source.groupby(block_keys, dropna=False):
    row = dict(zip(block_keys, keys))
    rel_sums = block.groupby("channel")["relative_power"].sum(min_count=EXPECTED_BANDS)
    rel_sum_ok = (rel_sums.notna() & rel_sums.sub(100).abs().lt(1e-6)).all() and len(rel_sums) == EXPECTED_CHANNELS
    unique_channel_band = block[["channel", "band"]].drop_duplicates().shape[0]
    row.update({
        "rows": len(block), "unique_channels": block["channel"].nunique(), "unique_bands": block["band"].nunique(),
        "unique_channel_band_pairs": unique_channel_band,
        "rows_ok": len(block) == EXPECTED_ROWS,
        "channels_ok": block["channel"].nunique() == EXPECTED_CHANNELS,
        "bands_ok": block["band"].nunique() == EXPECTED_BANDS and set(block["band"]) == EXPECTED_BAND_SET,
        "channel_band_pairs_ok": unique_channel_band == EXPECTED_ROWS,
        "missing_absolute": int(block["absolute_power"].isna().sum()),
        "missing_relative": int(block["relative_power"].isna().sum()),
        "duplicate_channel_band_keys": int(block.duplicated(["channel", "band"]).sum()),
        "relative_percent_sum_ok": bool(rel_sum_ok),
    })
    qc_rows.append(row)
block_qc = pd.DataFrame(qc_rows)
display(block_qc.head())

,subject_id,group,task,session,timepoint,run,condition,rows,unique_channels,unique_bands,unique_channel_band_pairs,rows_ok,channels_ok,bands_ok,channel_band_pairs_ok,missing_absolute,missing_relative,duplicate_channel_band_keys,relative_percent_sum_ok
0,Mc02,Control,Math,ses-1,PRE,1,Baseline,160,32,5,160,True,True,True,True,0,0,0,True
1,Mc02,Control,Math,ses-1,PRE,2,Activity1,160,32,5,160,True,True,True,True,0,0,0,True
2,Mc02,Control,Math,ses-1,PRE,3,Activity2,160,32,5,160,True,True,True,True,0,0,0,True
3,Mc02,Control,Math,ses-2,POST,1,Baseline,160,32,5,160,True,True,True,True,0,0,0,True
4,Mc02,Control,Math,ses-2,POST,2,Activity1,160,32,5,160,True,True,True,True,0,0,0,True


## 3. Average channels
For each participant × task × session × condition × band, average across the 32 channels. Channel SD is retained descriptively as spatial variability.

In [4]:
aggregation_keys = ["subject_id", "group", "task", "session", "timepoint", "run", "condition", "band"]
participant_band = source.groupby(aggregation_keys, as_index=False).agg(
    absolute_power_mean=("absolute_power", "mean"), absolute_power_sd_channels=("absolute_power", "std"),
    relative_power_mean=("relative_power", "mean"), relative_power_sd_channels=("relative_power", "std"),
    n_channels=("channel", "nunique"), n_rows=("channel", "size"),
)
participant_band["channels_complete"] = participant_band["n_channels"].eq(32) & participant_band["n_rows"].eq(32)
display(participant_band.head())

,subject_id,group,task,session,timepoint,run,condition,band,absolute_power_mean,absolute_power_sd_channels,relative_power_mean,relative_power_sd_channels,n_channels,n_rows,channels_complete
0,Mc02,Control,Math,ses-1,PRE,1,Baseline,alpha,1.091060e+05,5.720731e+05,3.861513,0.552056,32,32,True
1,Mc02,Control,Math,ses-1,PRE,1,Baseline,beta,9.140464e+04,4.797841e+05,3.014921,0.394379,32,32,True
2,Mc02,Control,Math,ses-1,PRE,1,Baseline,delta,3.084992e+06,1.634840e+07,84.940978,1.682379,32,32,True
3,Mc02,Control,Math,ses-1,PRE,1,Baseline,gamma,2.841447e+04,1.507264e+05,0.839169,0.187534,32,32,True
4,Mc02,Control,Math,ses-1,PRE,1,Baseline,theta,2.283194e+05,1.201830e+06,7.343419,0.597511,32,32,True


## 4. Calculate PRE–POST change
Changes are calculated for the same participant, task, run/condition, and band. Only observed paired PRE and POST values are subtracted; no imputation is performed.

In [5]:
change_keys = ["subject_id", "group", "task", "run", "condition", "band"]
wide = participant_band.pivot(index=change_keys, columns="timepoint", values=["absolute_power_mean", "relative_power_mean"])
wide.columns = ["_".join(str(x) for x in col if str(x)) for col in wide.columns]
wide = wide.reset_index()
for metric in ["absolute_power_mean", "relative_power_mean"]:
    pre, post = f"{metric}_PRE", f"{metric}_POST"
    if pre in wide.columns and post in wide.columns:
        wide[f"{metric}_change_POST_minus_PRE"] = wide[post] - wide[pre]
display(wide.head())

,subject_id,group,task,run,condition,band,absolute_power_mean_POST,absolute_power_mean_PRE,relative_power_mean_POST,relative_power_mean_PRE,absolute_power_mean_change_POST_minus_PRE,relative_power_mean_change_POST_minus_PRE
0,Mc02,Control,Math,1,Baseline,alpha,8.284848e+05,1.091060e+05,2.798556,3.861513,7.193788e+05,-1.062956
1,Mc02,Control,Math,1,Baseline,beta,1.031079e+06,9.140464e+04,3.277318,3.014921,9.396743e+05,0.262397
2,Mc02,Control,Math,1,Baseline,delta,4.645550e+07,3.084992e+06,84.616789,84.940978,4.337051e+07,-0.324189
3,Mc02,Control,Math,1,Baseline,gamma,3.682609e+05,2.841447e+04,1.158321,0.839169,3.398464e+05,0.319151
4,Mc02,Control,Math,1,Baseline,theta,2.362162e+06,2.283194e+05,8.149016,7.343419,2.133843e+06,0.805597


## 5. Summaries and output files

In [6]:
qc_summary = block_qc.groupby(["task", "group"], as_index=False).agg(
    blocks=("subject_id", "size"), all_160_rows=("rows_ok", "all"), all_32_channels=("channels_ok", "all"),
    all_5_bands=("bands_ok", "all"), all_160_unique_channel_band_pairs=("channel_band_pairs_ok", "all"),
    missing_absolute_power=("missing_absolute", "sum"), missing_relative_power=("missing_relative", "sum"),
    duplicate_keys=("duplicate_channel_band_keys", "sum"), relative_percent_sum_ok=("relative_percent_sum_ok", "all"),
)
qc_summary["all_qc_pass"] = qc_summary[["all_160_rows", "all_32_channels", "all_5_bands", "all_160_unique_channel_band_pairs", "relative_percent_sum_ok"]].all(axis=1) & qc_summary["missing_absolute_power"].eq(0) & qc_summary["missing_relative_power"].eq(0) & qc_summary["duplicate_keys"].eq(0)
coverage = participant_band.groupby(["task", "group", "timepoint", "condition", "band"], as_index=False).agg(
    participants=("subject_id", "nunique"), incomplete_channel_averages=("channels_complete", lambda x: int((~x).sum()))
)
paired_coverage = wide.groupby(["task", "group", "condition", "band"], as_index=False).agg(
    rows=("subject_id", "size"), n_with_PRE=("absolute_power_mean_PRE", "count"),
    n_with_POST=("absolute_power_mean_POST", "count"), n_paired_absolute=("absolute_power_mean_change_POST_minus_PRE", "count"),
    n_paired_relative=("relative_power_mean_change_POST_minus_PRE", "count")
)
outputs = {
    "eeg_channel_averaged_long.csv": participant_band,
    "eeg_channel_averaged_prepost_change.csv": wide,
    "eeg_channel_averaging_qc_by_block.csv": block_qc,
    "eeg_channel_averaging_qc_summary.csv": qc_summary,
    "eeg_aggregation_coverage.csv": coverage,
    "eeg_prepost_pair_coverage.csv": paired_coverage,
}
for filename, table in outputs.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)
    print(f"Saved {filename}: {len(table):,} rows")
display(qc_summary)
display(paired_coverage.head(20))

Saved eeg_channel_averaged_long.csv: 1,600 rows
Saved eeg_channel_averaged_prepost_change.csv: 800 rows
Saved eeg_channel_averaging_qc_by_block.csv: 320 rows
Saved eeg_channel_averaging_qc_summary.csv: 4 rows
Saved eeg_aggregation_coverage.csv: 100 rows
Saved eeg_prepost_pair_coverage.csv: 50 rows


,task,group,blocks,all_160_rows,all_32_channels,all_5_bands,all_160_unique_channel_band_pairs,missing_absolute_power,missing_relative_power,duplicate_keys,relative_percent_sum_ok,all_qc_pass
0,Math,Control,90,True,True,True,True,0,0,0,True,True
1,Math,Experiment,114,True,True,True,True,0,0,0,True,True
2,Reading,Control,48,True,True,True,True,0,0,0,True,True
3,Reading,Experiment,68,True,True,True,True,0,0,0,True,True


,task,group,condition,band,rows,n_with_PRE,n_with_POST,n_paired_absolute,n_paired_relative
0,Math,Control,Activity1,alpha,15,15,15,15,15
1,Math,Control,Activity1,beta,15,15,15,15,15
2,Math,Control,Activity1,delta,15,15,15,15,15
3,Math,Control,Activity1,gamma,15,15,15,15,15
4,Math,Control,Activity1,theta,15,15,15,15,15
5,Math,Control,Activity2,alpha,15,15,15,15,15
6,Math,Control,Activity2,beta,15,15,15,15,15
7,Math,Control,Activity2,delta,15,15,15,15,15
8,Math,Control,Activity2,gamma,15,15,15,15,15
9,Math,Control,Activity2,theta,15,15,15,15,15


## Interpretation / scope

- This produces a global 32-channel summary per participant, not regional or electrode-specific inference.
- Absolute and relative power remain separate. Bands and conditions are not collapsed.
- QC verifies the structure of exported power tables; it does not assess raw EEG artifacts or establish scientific effects.
- ROI aggregation is a separate next step and requires a verified electrode-to-region mapping from the montage file.